# AMPS II · Week 06 — Lab
## Structural dynamics (modes, Newmark, Rayleigh damping) and linear buckling with SLEPc

| | |
|---|---|
| **Lecture sections** | Part A (semi-discrete equations, Newmark, stability, Rayleigh) · Part B (SLEPc: frequencies, buckling) · Part C (verification) · Part D (LLM) |
| **Time** | ≈ 80 min in class |
| **You will** | test the Newmark family on one oscillator · compute and classify the modes of a cantilever · compare Newmark with the exact semi-discrete solution · measure a damping ratio · compute the buckling loads of a column and match them to Euler |

**Reminder.** `%%fenicsx` cells are separate MPI programs; ordinary cells have NumPy and matplotlib only — and `sdof.py`, which is NumPy only.
The three model modules are written once with `%%writefile` (§1). Results pass through `/tmp/fnx/`.

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

## 1 · The model modules  *(lecture §A2–A4, §B1–B3, §C1)*
`sdof.py`: the Newmark step on one oscillator — amplification matrix, spectral radius, period ratio, step response and its exact solution. `dyn.py`: the
plane-stress cantilever — $M$, $K$, the load and the tip functional, `modes` (SLEPc shift-invert, with the axial share of each mode), `static`, `newmark`
(PETSc vectors, one factorisation) and `dense_modal` (all modes, serial: the exact semi-discrete solution). `buckle.py`: the 3D column — pre-buckling
solve, $K_G$, and $(-K_G)\phi=\theta K\phi$. Run the three cells.

In [ ]:
%%writefile /tmp/fnx/sdof.py
"""The Newmark family on one oscillator  q'' + 2 zeta w q' + w^2 q = f  -- NumPy only.
One step maps (q, v, a)_n -> (q, v, a)_{n+1}; for f = 0 this is a 3 x 3 amplification matrix A(Omega = w dt).
Stability: spectral radius rho(A) <= 1. Accuracy: the angle of the complex eigenvalues of A vs Omega (period error)."""
import numpy as np

def amplification(Omega, beta=0.25, gamma=0.5, zeta=0.0):
    """3 x 3 amplification matrix of the Newmark step for q'' + 2 zeta w q' + w^2 q = 0, time in units of 1/w (w = 1)"""
    dt = Omega
    # implicit relation for a_{n+1}: a + 2 zeta (v_p + gamma dt a) + (q_p + beta dt^2 a) = 0
    den = 1 + 2 * zeta * gamma * dt + beta * dt**2
    A = np.zeros((3, 3))
    for j, e in enumerate(np.eye(3)):
        q, v, a = e
        qp = q + dt * v + dt**2 * (0.5 - beta) * a
        vp = v + dt * (1 - gamma) * a
        an = -(qp + 2 * zeta * vp) / den
        A[:, j] = [qp + beta * dt**2 * an, vp + gamma * dt * an, an]
    return A

def spectral_radius(Omega, **kw):
    return float(np.max(np.abs(np.linalg.eigvals(amplification(Omega, **kw)))))

def period_ratio(Omega, **kw):
    """numerical period / exact period for the undamped oscillator (from the phase of the principal eigenvalue)"""
    lam = np.linalg.eigvals(amplification(Omega, **kw))
    osc = lam[lam.imag > 1e-14]                     # the complex pair of the oscillation (the third eigenvalue is spurious)
    return Omega / np.angle(osc[0]) if len(osc) else np.inf

def integrate(w, dt, n, f=1.0, q0=0.0, v0=0.0, beta=0.25, gamma=0.5, zeta=0.0):
    """step response (constant f from t = 0+) with the consistent initial acceleration; returns t, q"""
    q, v = q0, v0
    a = f - 2 * zeta * w * v - w**2 * q
    out = [q]
    for _ in range(n):
        qp = q + dt * v + dt**2 * (0.5 - beta) * a
        vp = v + dt * (1 - gamma) * a
        a = (f - 2 * zeta * w * vp - w**2 * qp) / (1 + 2 * zeta * w * gamma * dt + w**2 * beta * dt**2)
        q, v = qp + beta * dt**2 * a, vp + gamma * dt * a
        out.append(q)
    return np.arange(n + 1) * dt, np.array(out)

def step_exact(w, t, f=1.0, zeta=0.0):
    """exact step response of the (damped) oscillator, q(0) = q'(0) = 0"""
    qs = f / w**2
    wd = w * np.sqrt(1 - zeta**2)
    return qs * (1 - np.exp(-zeta * w * t) * (np.cos(wd * t) + zeta / np.sqrt(1 - zeta**2) * np.sin(wd * t)))

In [ ]:
%%writefile /tmp/fnx/dyn.py
"""Elastodynamics of a 2D cantilever (plane stress, thickness 1) in DOLFINx 0.11 + SLEPc:
mass and stiffness matrices, natural frequencies (shift-invert), static solution, Newmark time integration with Rayleigh damping
(PETSc vectors, the effective matrix factorised ONCE), and -- serially, for small meshes -- the exact semi-discrete solution by
complete modal superposition (dense matrices)."""
import numpy as np
import ufl
from mpi4py import MPI
from petsc4py import PETSc
from slepc4py import SLEPc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import apply_lifting, assemble_matrix, assemble_vector, set_bc

comm = MPI.COMM_WORLD
def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

BETA_L = [1.875104069, 4.694091133, 7.854757438, 10.99554073]       # cantilever: beta_n L of Euler-Bernoulli

def lu(ksp):
    ksp.setType("preonly"); ksp.getPC().setType("lu"); ksp.getPC().setFactorSolverType("mumps")
    return ksp

class Cantilever:
    """[0, L] x [0, H], clamped at x = 0, a shear traction of total P (per unit thickness) on x = L, downwards.
    plane = "stress" (a thin beam: lambda* = E nu / (1 - nu^2)) or "strain"."""
    def __init__(self, L, H, nx, ny, E, nu, rho, P=1.0, plane="stress", deg=2):
        self.L, self.H, self.E, self.nu, self.rho, self.P = L, H, E, nu, rho, P
        msh = self.msh = mesh.create_rectangle(comm, [[0, 0], [L, H]], [nx, ny], cell_type=mesh.CellType.triangle)
        mu = E / (2 * (1 + nu)); lm = E * nu / (1 - nu**2) if plane == "stress" else E * nu / ((1 + nu) * (1 - 2 * nu))
        self.mu = mu
        V = self.V = fem.functionspace(msh, ("Lagrange", deg, (2,)))
        u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
        eps = lambda w: ufl.sym(ufl.grad(w))
        self.sig = lambda w: lm * ufl.tr(eps(w)) * ufl.Identity(2) + 2 * mu * eps(w)
        left = mesh.locate_entities_boundary(msh, 1, lambda x: np.isclose(x[0], 0))
        right = np.sort(mesh.locate_entities_boundary(msh, 1, lambda x: np.isclose(x[0], L)))
        self.ds = ufl.Measure("ds", domain=msh, subdomain_data=mesh.meshtags(msh, 1, right, np.ones(len(right), dtype=np.int32)))
        self.bc = fem.dirichletbc(np.zeros(2), fem.locate_dofs_topological(V, 1, left), V)
        self.mf = fem.form(rho * ufl.inner(u, v) * ufl.dx)
        self.kf = fem.form(ufl.inner(self.sig(u), eps(v)) * ufl.dx)
        self.K = assemble_matrix(self.kf, bcs=[self.bc], diag=1.0); self.K.assemble()     # BC rows: identity
        self.M = assemble_matrix(self.mf, bcs=[self.bc], diag=0.0); self.M.assemble()     # BC rows: zero -> BC "modes" at infinity
        self.M1 = assemble_matrix(self.mf, bcs=[self.bc], diag=1.0); self.M1.assemble()   # BC rows: identity (time stepping, a0)
        self.f = assemble_vector(fem.form(ufl.dot(fem.Constant(msh, np.array([0.0, -P / H])), v) * self.ds(1)))
        apply_lifting(self.f, [self.kf], [[self.bc]]); self.f.ghostUpdate(PETSc.InsertMode.ADD, PETSc.ScatterMode.REVERSE); set_bc(self.f, [self.bc])
        self.g = assemble_vector(fem.form(v[1] / H * self.ds(1)))                       # tip = mean u_y on x = L, as a linear functional
        self.g.ghostUpdate(PETSc.InsertMode.ADD, PETSc.ScatterMode.REVERSE); set_bc(self.g, [self.bc])
        self.Ix, self.A = H**3 / 12, H

    def tip(self, x):
        """mean u_y on x = L of a PETSc vector of displacements"""
        return self.g.dot(x)

    def euler_bernoulli(self, n):
        return BETA_L[n - 1]**2 * np.sqrt(self.E * self.Ix / (self.rho * self.A * self.L**4))

    def static(self):
        u = fem.Function(self.V); k = lu(PETSc.KSP().create(comm)); k.setOperators(self.K)
        k.solve(self.f, u.x.petsc_vec); u.x.scatter_forward()
        return u

    def modes(self, n=6):
        """the n lowest natural frequencies (rad/s), M-normalised mode vectors, and the share of axial motion of each mode"""
        e = SLEPc.EPS().create(comm); e.setOperators(self.K, self.M); e.setProblemType(SLEPc.EPS.ProblemType.GHEP)
        e.setDimensions(n); e.setTarget(0.0); e.setWhichEigenpairs(SLEPc.EPS.Which.TARGET_MAGNITUDE)
        st = e.getST(); st.setType("sinvert"); lu(st.getKSP()); e.setTolerances(1e-10, 500); e.solve()
        out = []
        for i in range(e.getConverged()):
            x = self.K.createVecRight(); lam = e.getEigenpair(i, x).real
            Mx = self.M.createVecRight(); self.M.mult(x, Mx); x.scale(1 / np.sqrt(x.dot(Mx)))
            ux, uy = x.array_r[0::2], x.array_r[1::2]
            ax = comm.allreduce(float(ux @ ux), op=MPI.SUM); tot = ax + comm.allreduce(float(uy @ uy), op=MPI.SUM)
            out.append((np.sqrt(lam), x, ax / tot))
        out.sort(key=lambda r: r[0])
        return out[:n]

    def newmark(self, dt, nsteps, beta=0.25, gamma=0.5, aM=0.0, aK=0.0, load=1.0, u0=None, record=()):
        """M a + C v + K u = load * f,  C = aM M + aK K.  Starts from u0 (PETSc vector or None = 0), v = 0 and the CONSISTENT
        acceleration a0 = M^-1 (load f - K u0). The matrix M + gamma dt C + beta dt^2 K is assembled and factorised once.
        record: extra vectors y; returns a dict with t, tip, energy (kinetic + strain - load work) and y . (M u) for each y."""
        M, K, f = self.M, self.K, self.f
        A = self.M1.copy(); A.axpy(gamma * dt * aM, M); A.axpy(gamma * dt * aK + beta * dt**2, K)   # BC rows: identity-dominated (also for beta = 0)
        ksp = lu(PETSc.KSP().create(comm)); ksp.setOperators(A)
        u, v, a, up, vp, r, tmp = (f.duplicate() for _ in range(7))
        u.set(0.0); v.set(0.0)
        if u0 is not None:
            u0.copy(u)
        r = f.copy(); r.scale(load); K.mult(u, tmp); r.axpy(-1.0, tmp); set_bc(r, [self.bc], alpha=0.0)
        k0 = lu(PETSc.KSP().create(comm)); k0.setOperators(self.M1); k0.solve(r, a)
        hist = {"t": [0.0], "tip": [self.tip(u)], "energy": [self._energy(u, v, load)], "rec": [[y.dot(self._Mx(u)) for y in record]]}
        for n in range(1, nsteps + 1):
            u.copy(up); up.axpy(dt, v); up.axpy(dt**2 * (0.5 - beta), a)
            v.copy(vp); vp.axpy(dt * (1 - gamma), a)
            r = f.copy(); r.scale(load)
            K.mult(up, tmp); r.axpy(-1.0, tmp)
            if aM or aK:
                M.mult(vp, tmp); r.axpy(-aM, tmp); K.mult(vp, tmp); r.axpy(-aK, tmp)
            set_bc(r, [self.bc], alpha=0.0)
            ksp.solve(r, a)
            up.copy(u); u.axpy(beta * dt**2, a)
            vp.copy(v); v.axpy(gamma * dt, a)
            hist["t"].append(n * dt); hist["tip"].append(self.tip(u)); hist["energy"].append(self._energy(u, v, load))
            hist["rec"].append([y.dot(self._Mx(u)) for y in record])
        return {k: np.array(x) for k, x in hist.items()}

    def _Mx(self, x):
        y = x.duplicate(); self.M.mult(x, y); return y

    def _energy(self, u, v, load):
        Kv = u.duplicate(); self.K.mult(u, Kv)
        return 0.5 * v.dot(self._Mx(v)) + 0.5 * u.dot(Kv) - load * self.f.dot(u)

    def dense_modal(self):
        """SERIAL only: all modes from the dense matrices (BC rows removed). Returns omega, the tip and load projections, so that
        the exact semi-discrete step response is  tip(t) = sum_i g_i c_i / omega_i^2 (1 - cos omega_i t)."""
        from scipy.linalg import eigh
        assert comm.size == 1, "dense_modal needs -np 1"
        Kd, Md = PETSc.Mat(), PETSc.Mat()
        self.K.convert("dense", Kd); self.M.convert("dense", Md)                         # into NEW matrices: convert() without out= is in place
        Kd, Md = Kd.getDenseArray(), Md.getDenseArray()
        free = np.setdiff1d(np.arange(Kd.shape[0]), self.bc.dof_indices()[0])
        lam, Phi = eigh(Kd[np.ix_(free, free)], Md[np.ix_(free, free)])
        return np.sqrt(lam), Phi.T @ self.g.array[free], Phi.T @ self.f.array[free]

In [ ]:
%%writefile /tmp/fnx/buckle.py
"""Linear buckling of a 3D column (hexahedra) in DOLFINx 0.11 + SLEPc: pre-buckling stress under a reference load, the geometric
stiffness  K_G = int sigma0 : (grad u . grad v^T),  and the eigenproblem  K phi = lambda (-K_G) phi,  solved as
(-K_G) phi = (1/lambda) K phi  with the positive definite K as the B matrix (no special treatment of the BC rows needed)."""
import numpy as np
import ufl
from mpi4py import MPI
from petsc4py import PETSc
from slepc4py import SLEPc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import apply_lifting, assemble_matrix, assemble_vector, set_bc

comm = MPI.COMM_WORLD
def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def lu(ksp):
    ksp.setType("preonly"); ksp.getPC().setType("lu"); ksp.getPC().setFactorSolverType("mumps")
    return ksp

class Column:
    """[0, b] x [0, h] x [0, L], clamped at z = 0, a uniform compressive traction P_ref / (b h) on z = L (free end).
    Bending with deflection in y uses I = b h^3/12 (the weak axis if h < b), deflection in x uses I = h b^3/12."""
    def __init__(self, b, h, L, nx, ny, nz, E, nu, deg=2, kg="full"):
        self.b, self.h, self.L, self.E, self.nu = b, h, L, E, nu
        msh = self.msh = mesh.create_box(comm, [[0, 0, 0], [b, h, L]], [nx, ny, nz], mesh.CellType.hexahedron)
        mu, lm = E / (2 * (1 + nu)), E * nu / ((1 + nu) * (1 - 2 * nu)); self.G = mu
        V = self.V = fem.functionspace(msh, ("Lagrange", deg, (3,)))
        u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
        eps = lambda w: ufl.sym(ufl.grad(w))
        sig = lambda w: lm * ufl.tr(eps(w)) * ufl.Identity(3) + 2 * mu * eps(w)
        base = mesh.locate_entities_boundary(msh, 2, lambda x: np.isclose(x[2], 0))
        top = np.sort(mesh.locate_entities_boundary(msh, 2, lambda x: np.isclose(x[2], L)))
        ds = ufl.Measure("ds", domain=msh, subdomain_data=mesh.meshtags(msh, 2, top, np.ones(len(top), dtype=np.int32)))
        self.bc = fem.dirichletbc(np.zeros(3), fem.locate_dofs_topological(V, 2, base), V)
        self.Pref = self.euler(1, "y")                                    # reference load: the first Euler load (lambda ~ 1)
        kf = fem.form(ufl.inner(sig(u), eps(v)) * ufl.dx)
        self.K = assemble_matrix(kf, bcs=[self.bc], diag=1.0); self.K.assemble()
        fv = assemble_vector(fem.form(ufl.dot(fem.Constant(msh, np.array([0.0, 0.0, -self.Pref / (b * h)])), v) * ds(1)))
        apply_lifting(fv, [kf], [[self.bc]]); fv.ghostUpdate(PETSc.InsertMode.ADD, PETSc.ScatterMode.REVERSE); set_bc(fv, [self.bc])
        self.u0 = fem.Function(V); k = lu(PETSc.KSP().create(comm)); k.setOperators(self.K); k.solve(fv, self.u0.x.petsc_vec)
        self.u0.x.scatter_forward()
        s0 = sig(self.u0)
        if kg == "full":
            gform = ufl.inner(s0, ufl.dot(ufl.grad(u), ufl.grad(v).T))                 # sigma_ij u_k,i v_k,j
        elif kg == "zz":
            gform = s0[2, 2] * ufl.dot(u.dx(2), v.dx(2))                               # sigma_zz u_k,z v_k,z (all k)
        else:
            gform = s0[2, 2] * u[2].dx(2) * v[2].dx(2)                                 # only k = z
        self.mKG = assemble_matrix(fem.form(-gform * ufl.dx), bcs=[self.bc], diag=0.0); self.mKG.assemble()
        self.ndofs = V.dofmap.index_map.size_global * 3

    def euler(self, n, axis):
        """Euler load of a clamped-free column, mode n, deflection along axis 'y' (I = b h^3/12) or 'x' (I = h b^3/12)"""
        I = self.b * self.h**3 / 12 if axis == "y" else self.h * self.b**3 / 12
        return (2 * n - 1)**2 * np.pi**2 * self.E * I / (4 * self.L**2)

    def solve(self, nev=6):
        """the nev smallest load factors lambda (P_cr = lambda P_ref) with their modes and the dominant deflection direction"""
        e = SLEPc.EPS().create(comm); e.setOperators(self.mKG, self.K); e.setProblemType(SLEPc.EPS.ProblemType.GHEP)
        e.setDimensions(nev); e.setWhichEigenpairs(SLEPc.EPS.Which.LARGEST_REAL)       # theta = 1/lambda: largest theta = smallest lambda
        lu(e.getST().getKSP()); e.setTolerances(1e-10, 1000); e.solve()
        out = []
        for i in range(e.getConverged()):
            x = self.K.createVecRight(); th = e.getEigenpair(i, x).real
            if th <= 0:
                continue
            a = x.array_r
            nx, ny = (comm.allreduce(float(a[c::3] @ a[c::3]), op=MPI.SUM) for c in (0, 1))
            out.append((1 / th, "y" if ny > nx else "x", x))
        out.sort(key=lambda r: r[0])
        return out[:nev], e.getConverged()

## 2 · One oscillator  *(lecture §A2–A3)*
Spectral radius and period ratio of four Newmark schemes, the central-difference limit, and a step response against the exact one.

In [ ]:
import sys; sys.path.insert(0, "/tmp/fnx")
import numpy as np
from sdof import *
for name, kw in (("average acceleration", dict(beta=0.25, gamma=0.5)), ("linear acceleration", dict(beta=1 / 6, gamma=0.5)),
                 ("central difference", dict(beta=0.0, gamma=0.5)), ("damped (0.3025, 0.6)", dict(beta=0.3025, gamma=0.6))):
    print(f"{name:22s}: rho(Omega = 1) = {spectral_radius(1.0, **kw):.4f}, rho(10) = {spectral_radius(10.0, **kw):.4f};  "
          f"period ratio at T/dt = 25: {period_ratio(2 * np.pi / 25, **kw):.5f}")
print(f"central difference: rho(1.99) = {spectral_radius(1.99, beta=0):.4f}, rho(2.01) = {spectral_radius(2.01, beta=0):.4f};  Omega^2/12 at T/dt = 25: {(2 * np.pi / 25)**2 / 12:.5f}")
w = 2 * np.pi
for n in (25, 50, 100):
    t, q = integrate(w, 1 / n, 2 * n)
    print(f"step response, T/dt = {n:3d}: max |q - exact| / q_static = {np.abs(q - step_exact(w, t)).max() * w**2:.2e},  peak {q.max() * w**2:.5f}")
t, q = integrate(w, 1 / 100, 300, zeta=0.05)
print(f"damped, zeta = 5 %: peak / static = {q.max() * w**2:.5f}   (1 + exp(-zeta pi / sqrt(1 - zeta^2)) = {1 + np.exp(-0.05 * np.pi / np.sqrt(1 - 0.0025)):.5f})")

> **✏️ Your turn (5 min).** (i) Why does the error of the step response fall by 4 when $\Delta t$ is halved, and why is the peak not exactly 2? (ii) The cantilever below has
> $\omega_{\max}\approx2\times10^6$ rad/s and $T_1=0.012$ s. Which time step would central difference need, and how many steps for one period $T_1$?

> *your answer*

## 3 · The cantilever: static deflection and modes  *(lecture §A1, §B1–B2)*
$L=1$, $H=0.1$ m, plane stress, $E=210$ GPa, $\nu=0.3$, $\rho=7800$ kg/m³, clamped at $x=0$, tip load $P=1000$ N/m. P2 on $20\times4$ cells.

In [ ]:
%%fenicsx -np 2
import sys; sys.path.insert(0, "/tmp/fnx")
from dyn import *
L, H, E, nu, rho, P = 1.0, 0.1, 210e9, 0.3, 7800.0, 1000.0      # SI; plane stress, thickness 1; tip load P [N/m]

cb = Cantilever(L, H, 20, 4, E, nu, rho, P)
us = cb.static(); ust = cb.tip(us.x.petsc_vec)
EI = E * H**3 / 12
say(f"static tip: FEM {ust:.6e};  Euler-Bernoulli {-P * L**3 / (3 * EI):.6e};  + shear {-P * L**3 / (3 * EI) - P * L / (5 / 6 * cb.mu * H):.6e}")
for i, (w, x, ax) in enumerate(cb.modes(6), 1):
    say(f"mode {i}: omega = {w:10.2f} rad/s,  axial share {ax:.3f}")
say("Euler-Bernoulli bending 1-4: " + ", ".join(f"{cb.euler_bernoulli(n):.2f}" for n in (1, 2, 3, 4)) + f";  axial mode, exact: {np.pi / (2 * L) * np.sqrt(E / rho):.2f} rad/s")

> **✏️ Your turn (5 min).** Label every mode (bending 1, 2, 3, axial). Compare the bending frequencies with Euler–Bernoulli **by shape, not by index**. Why do they fall
> further below it for higher modes?

> *your answer*

## 4 · Newmark against the exact semi-discrete solution  *(lecture §A1–A2, §C2–C3)*
Serially, `dense_modal()` diagonalises the full $M$, $K$ (720 unknowns): the exact response of the **discrete** system to the step load. Compare Newmark (average
acceleration) with it, for the tip and for the coordinate of mode 1, and watch the energy.

In [ ]:
%%fenicsx -np 1
import sys; sys.path.insert(0, "/tmp/fnx")
from dyn import *
L, H, E, nu, rho, P = 1.0, 0.1, 210e9, 0.3, 7800.0, 1000.0      # SI; plane stress, thickness 1; tip load P [N/m]

cb = Cantilever(L, H, 20, 4, E, nu, rho, P)
us = cb.static(); ust = cb.tip(us.x.petsc_vec)
om, gi, ci = cb.dense_modal()
exact = lambda t: np.array([np.sum(gi * ci / om**2 * (1 - np.cos(om * x))) for x in np.atleast_1d(t)])
w1, phi1, _ = cb.modes(1)[0]; q1s = phi1.dot(cb.f) / w1**2; T1 = 2 * np.pi / w1
tt = np.linspace(0, 2 * T1, 4001)
say(f"{len(om)} modes, omega_1 = {om[0]:.2f}, omega_max = {om.max():.3e};  exact peak tip / static = {exact(tt).min() / ust:.4f}")
for n in (25, 50, 100):
    dt = T1 / n; r = cb.newmark(dt, 2 * n, record=[phi1])
    e1 = np.abs(r["rec"][:, 0] - q1s * (1 - np.cos(w1 * r["t"]))).max() / abs(q1s)
    et = np.abs(r["tip"] - exact(r["t"])).max() / abs(ust)
    say(f"T1/dt = {n:3d}: mode-1 error {e1:.2e},  tip error {et:.2e},  peak tip / static {r['tip'].min() / ust:.4f},  "
        f"energy drift {np.ptp(r['energy']) / abs(cb.f.dot(us.x.petsc_vec)):.1e},  omega_max dt = {om.max() * dt:.0f}")

> **✏️ Your turn (5 min).** The mode-1 error falls by 4 per halving; the tip error by less than 2. Explain. Would a smaller $\Delta t$ ever make the tip error second order here?

> *your answer*

## 5 · Rayleigh damping  *(lecture §A4)*
$\zeta=5\,\%$ in mode 1: $\alpha_M=\zeta\omega_1$, $\alpha_K=\zeta/\omega_1$. Measure the damping ratio from the logarithmic decrement of the mode-1 coordinate, and the damping that
the higher modes receive.

In [ ]:
%%fenicsx -np 2
import sys; sys.path.insert(0, "/tmp/fnx")
from dyn import *
L, H, E, nu, rho, P = 1.0, 0.1, 210e9, 0.3, 7800.0, 1000.0      # SI; plane stress, thickness 1; tip load P [N/m]

cb = Cantilever(L, H, 20, 4, E, nu, rho, P)
us = cb.static(); ust = cb.tip(us.x.petsc_vec)
ms = cb.modes(4); w1, phi1 = ms[0][0], ms[0][1]; q1s = phi1.dot(cb.f) / w1**2; T1 = 2 * np.pi / w1
z = 0.05
r = cb.newmark(T1 / 100, 1000, aM=z * w1, aK=z / w1, record=[phi1])
q = r["rec"][:, 0] / q1s
pk = [i for i in range(1, len(q) - 1) if q[i] > q[i - 1] and q[i] >= q[i + 1]]
delta = np.log((q[pk[0]] - 1) / (q[pk[1]] - 1))
say(f"mode-1 peak / static {q.max():.5f};  log decrement {delta:.5f} -> zeta = {delta / np.sqrt(4 * np.pi**2 + delta**2):.5f};  tip peak / static {r['tip'].min() / ust:.4f}")
say("zeta of modes 1-4: " + ", ".join(f"{z / 2 * (w1 / w + w / w1):.3f}" for w, _, _ in ms))

> **✏️ Your turn (3 min).** Why is the tip peak (1.83) below the mode-1 peak (1.85)? If you wanted 5 % in modes 1 **and** 2, which $\alpha_M$, $\alpha_K$?

> *your answer*

## 6 · Linear buckling of the column  *(lecture §B3–B5)*
Clamped at $z=0$, compressed at $z=L$; $b\times h\times L=0.05\times0.025\times1$ m; reference load = the first Euler load, so $\lambda$ = 1, 4, 9, 25, 36, 49 in theory.
P2 hexahedra $4\times4\times40$, and P1 for comparison.

In [ ]:
%%fenicsx -np 2
import sys; sys.path.insert(0, "/tmp/fnx")
from buckle import *
b, h, Lc, E, nu = 0.05, 0.025, 1.0, 210e9, 0.3                  # column b x h x L [m]

euler = {("y", 1): 1, ("x", 1): 4, ("y", 2): 9, ("y", 3): 25, ("x", 2): 36, ("y", 4): 49}
for deg, mesh_ in ((2, (4, 4, 40)), (1, (4, 4, 30))):
    c = Column(b, h, Lc, *mesh_, E, nu, deg=deg)
    if deg == 2:
        uz = comm.allreduce(float(c.u0.x.array[2::3].min()) if c.u0.x.array.size else 0.0, op=MPI.MIN)
        say(f"pre-buckling u_z(L) = {uz:.5e}  (-P L / (E A) = {-c.Pref * Lc / (E * b * h):.5e})")
    res, nconv = c.solve(6)
    say(f"P{deg} {mesh_}, {c.ndofs} DOFs: " + ", ".join(f"{l:.4f} ({a})" for l, a, _ in res))

> **✏️ Your turn (5 min).** Match each P2 load factor to an Euler mode **by its deflection direction** ($x$ or $y$) and its order. Which ones lie above Euler, which below, and why?
> What happened with P1?

> *your answer*

## 7 · LLM exercise: natural frequencies of the column  *(lecture Part D)*
Ask your LLM for a cell that uses `buckle.py` (paste it and the API sheet §16) to compute the first natural frequencies of the **unloaded** column and compare
them with Euler–Bernoulli for bending in $x$ and in $y$ ($\rho=7800$ kg/m³).

```text
Using the attached module buckle.py (DOLFINx 0.11, SLEPc), write a %%fenicsx -np 2 cell that builds Column(0.05, 0.025, 1.0, 4, 4, 40, 210e9, 0.3)
(P2 hexahedra, clamped at z = 0), assembles the mass matrix rho * inner(u, v) * dx on c.V with the same Dirichlet condition, and computes the 6 lowest
natural frequencies with SLEPc (shift-invert). Classify each mode by its dominant displacement component (x, y, z or torsion) and compare the bending
modes with the Euler-Bernoulli cantilever omega = (beta_n L)^2 sqrt(E I / (rho A L^4)), beta_1 L = 1.8751, beta_2 L = 4.6941.
```

```python
# paste here
```

**Check before you trust it**

- [ ] $K$ comes with `diag=1` and $M$ with `diag=0` on the clamped DOFs (or the constrained DOFs are removed): no spurious frequencies near the target
- [ ] The modes are classified by shape; torsion (rotation of the section) is recognised as such and not compared with bending
- [ ] $I$ is chosen by the bending direction: deflection in $y$ uses $bh^3/12$
- [ ] The mesh is refined once: the bending frequencies change in the 4th digit

In [ ]:
%%fenicsx -np 2
# TODO: the LLM's cell (after you checked it)

> *your verdict on the LLM's code*